In [18]:
import os
import pandas as pd
import requests
import re
from tqdm import tqdm

os.chdir ('C:\\Users\\Admin\\Documents\\Hanna')

In [20]:

# Function to clean scientific names by removing authority
def clean_scientific_name(name):
    """
    Cleans a scientific name by removing authority and retaining only the genus
    or genus + species, depending on the input.
    - For genus-level names: Keep only the first word.
    - For species-level names: Keep the first two words (genus + species).
    """
    if pd.isna(name) or not isinstance(name, str):
        return name  # Return as-is if the input is invalid
    
    # Match the first two words (genus and species) or just the first word (genus)
    match = re.match(r"^\s*([A-Za-z]+(?:\s+[a-z]+)?)", name)
    if match:
        return match.group(1).strip()  # Return the matched genus or genus + species
    
    return name  # Return the original name if no match is found
# Function to query WoRMS API for AphiaID and ranks
def get_aphia_info(taxa_list):
    results = []
    
    for taxon in tqdm(taxa_list, desc="Querying WoRMS"):
        try:
            # Clean the taxon name to remove authority
            cleaned_taxon = clean_scientific_name(taxon)
            
            # Query the WoRMS API (without marine_only filter to include non-marine taxa)
            url = f"https://www.marinespecies.org/rest/AphiaRecordsByName/{cleaned_taxon}"
            response = requests.get(url)
            
            if response.status_code == 200:
                records = response.json()
                if records:  # If records are found
                    record = records[0]  # Take only the first match
                    results.append({
                        "AphiaID": record.get("AphiaID"),
                        "scientificname": record.get("scientificname"),
                        "aphiaRank": record.get("rank"),
                        "Input_Name": taxon,  # Original input name with authority
                        "Cleaned_Name": cleaned_taxon  # Cleaned name used for querying
                    })
                else:
                    # No records found
                    results.append({"Input_Name": taxon, "Cleaned_Name": cleaned_taxon, "AphiaID": None, "scientificname": None, "aphiaRank": None})
            else:
                # API error
                results.append({"Input_Name": taxon, "Cleaned_Name": cleaned_taxon, "AphiaID": None, "scientificname": None, "aphiaRank": None})
        except Exception as e:
            # Handle any exceptions
            print(f"Error querying taxon '{taxon}': {e}")
            results.append({"Input_Name": taxon, "Cleaned_Name": cleaned_taxon, "AphiaID": None, "scientificname": None, "aphiaRank": None})
    
    return results
# Function to query GBIF API for scientific names and ranks
def get_gbif_data(gbif_ids):
    gbif_base_url = "https://api.gbif.org/v1/species/"
    results = []
    
    for gbif_id in tqdm(gbif_ids, desc="Querying GBIF"):
        try:
            url = f"{gbif_base_url}{gbif_id}"
            response = requests.get(url)
            
            if response.status_code == 200:
                data = response.json()
                results.append({
                    "gbifID": gbif_id,
                    "ScientificName": data.get("scientificName"),
                    "gbifRank": data.get("rank")
                })
            else:
                results.append({"gbifID": gbif_id, "ScientificName": None, "gbifRank": None})
        except Exception as e:
            print(f"Error querying GBIF ID '{gbif_id}': {e}")
            results.append({"gbifID": gbif_id, "ScientificName": None, "gbifRank": None})
    
    return results
# Example: Taxa lists (replace with your actual IDs)
exclude_classes = [212, 359, 131, 367, 360, 10713444, 361, 216, 253, 11592253, 11374670, 7742773, 11210652, 11186032, 9273948, 144, 7188530, 11733052, 
                   176, 121, 140, 120, 7774442, 224, 254, 11881065, 352, 11418114, 207, 119, 11500725, 11377931, 11569602, 281, 7375758, 139, 355]
exclude_orders = [1313, 549, 590, 587, 588, 883, 9697017, 1441, 1456, 1068, 1005, 1306, 550, 837, 495, 473, 774, 538, 548, 1153, 426, 497, 544, 1305, 
                  12261528, 1307, 1077, 8403901, 499, 9341894, 773, 890, 888, 1103, 1166, 498, 659, 1312, 1308, 772, 9311931, 1311, 1067, 494, 782, 1310, 
                  537, 12216105, 1107, 1106, 9463058, 708, 505, 1163, 1069, 11205145, 901, 11125512, 1165, 547, 1167, 12233564, 496, 11131731, 12182793, 
                  771, 11164857, 679, 687, 9445058, 693, 694, 695, 1417, 11443919, 1428, 9697017, 868]
exclude_families = [7990, 4453, 7996, 4455, 8008, 4428, 4449, 8001, 2073, 6103, 4901, 4902, 4303274, 4960, 9044, 5048, 9052, 6207, 4461, 9054, 9055, 3589]
exclude_genus = [2312692, 2316452, 2312497, 2316610, 2114424, 2115369, 2269480, 2219967, 2115030, 6180409, 2406056, 10816205, 2269549, 2115338, 2114220, 
                 2114718, 2115124, 2115195, 7927564, 2269491, 5431531, 2234785, 7544733, 2506123, 2114333, 2307067, 2228004, 2332495, 2115284, 2269508, 
                 2114265, 2114807, 2114924, 2111272, 2332548, 2307434, 2115460, 2114350, 2114914, 2115157, 2306892, 2114649, 2114705, 2114882, 2114837, 
                 2227984, 4322151, 2115436]
exclude_species = [2114445, 2114439, 2115377, 2115373, 2115371, 2115375, 2269484, 4282555, 2219970, 2115037, 2115035, 6180409, 2406056, 5233140, 4987686, 
                   4987684, 2234694, 2269550, 2269551, 2266609, 2115346, 2115344, 5166399, 5166402, 5166405, 2114723, 2115141, 2115138, 2115131, 2115142, 
                   2115128, 2115197, 2115202, 2115200, 5193700, 1028577, 1028479, 1028572, 1028559, 2269492, 2269495, 2264841, 2264478, 5717325, 2264800, 
                   4334397, 1028478, 5166387, 2115074, 5166468, 4333480, 2307068, 2307078, 2114329, 2332511, 2332507, 2115300, 2115324, 2115303, 2115325, 
                   2115305, 5185508, 2115009, 2115018, 5166441, 5166453, 2269510, 4331035, 2222553, 2222554, 5177646, 5177649, 2115262, 5717403, 5718524, 
                   5165879, 2114278, 2114282, 2114270, 2114284, 8217523, 2227978, 5233166, 2114824, 11082687, 2114817, 2114926, 2114925, 2111274, 2114872, 
                   4334013, 2114874, 2114873, 7661024, 2264798, 2114956, 2332555, 2332553, 2332557, 4356840, 5166645, 5166639, 2307435, 5165790, 5165820, 
                   5165822, 5165823, 5165827, 5165810, 5165819, 5165809, 5165793, 5165784, 2115466, 11602113, 2114417, 2114406, 2114351, 2114401, 2265711, 
                   2115523, 2316453, 5185412, 4334981, 2312501, 2265692, 1028563, 2114923, 2114921, 2114916, 2114917, 2115159, 2307011, 4339887, 2265599, 
                   5193904, 5193937, 4334696, 2114661, 2114668, 2114663, 4334690, 2217381, 2114711, 8166174, 2269500, 2269434, 2114890, 2114884, 2114900, 
                   2114894, 2114895, 2114902, 2114886, 2114903, 2114897, 5166379, 2114838, 5177644, 5177643, 5177641, 2227991, 2227990, 2227985, 2312723, 
                   4322171, 4322180, 9483153, 2316612, 2115437, 4335010, 6467075, 4335044, 6467074, 2216393, 2219836, 2219631, 2219636]
# Combine all taxa into a single list
all_taxa = exclude_classes + exclude_orders + exclude_families + exclude_genus + exclude_species
# Step 1: Query GBIF for scientific names and ranks
gbif_data = get_gbif_data(all_taxa)
gbif_df = pd.DataFrame(gbif_data)
# Step 2: Query WoRMS for AphiaIDs and ranks
if "ScientificName" in gbif_df.columns:
    taxa_list = gbif_df["ScientificName"].dropna().tolist()  # Remove any NA values
    worms_results = get_aphia_info(taxa_list)
    worms_df = pd.DataFrame(worms_results)
else:
    print("No scientific names found in GBIF data.")
    worms_df = pd.DataFrame()
# Step 3: Combine GBIF and WoRMS data
if not worms_df.empty:
    final_data = pd.merge(gbif_df, worms_df, left_on="ScientificName", right_on="Input_Name", how="left")
else:
    final_data = gbif_df
# Step 4: Save to Excel
final_data.to_excel("excluded_taxa_with_ranks.xlsx", index=False)
# Print the final data
print(final_data)

Querying WoRMS: 100%|██████████| 363/363 [03:27<00:00,  1.75it/s]


      gbifID                                    ScientificName gbifRank  \
0        212                                              Aves    CLASS   
1        359                                          Mammalia    CLASS   
2        131                                          Amphibia    CLASS   
3        367                                         Arachnida    CLASS   
4        360                                         Chilopoda    CLASS   
..       ...                                               ...      ...   
364  6467074  Xanthocalanus spinodenticulatus Markhaseva, 1998  SPECIES   
365  2216393                 Apherusa glacialis (Hansen, 1888)  SPECIES   
366  2219836                  Gammarus wilkitzkii Birula, 1897  SPECIES   
367  2219631               Onisimus glacialis (G.O.Sars, 1900)  SPECIES   
368  2219636                 Onisimus nanseni (G.O.Sars, 1900)  SPECIES   

      AphiaID                   scientificname aphiaRank  \
0      1836.0                          

## Go through manually and fill in the aphiaIDs that are missing